# VisDrone Real Dataset Fast Smoke Test

Очень быстрый тест основных модулей на настоящем VisDrone YOLO dataset. Ноутбук берёт маленький subset из `configs/project_config.yaml -> dataset.root`, затем проверяет validation, analyzer, rule engine, object bank, custom transforms и COCOeval. Обучение YOLO не запускается.

Для проверки evaluator используются prediction-файлы, созданные из ground truth с confidence. Это намеренно: цель ноутбука — проверить работоспособность модулей проекта за минуты, а не качество модели.

In [ ]:
from pathlib import Path
import json, sys, time

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / 'src').exists() else cwd.parent
assert (PROJECT_ROOT / 'src').exists(), f'Cannot find project root from {cwd}'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
from src.analysis.dataset_analyzer import DatasetAnalyzerConfig, analyze_dataset
from src.augmentation.albumentations_transforms import apply_custom_transforms
from src.augmentation.object_bank import ObjectBank
from src.augmentation.policy_to_ultralytics import AugmentationPolicy
from src.data.subset_builder import build_yolo_subset
from src.data.visdrone_fixture import create_predictions_from_gt
from src.data.visdrone_manager import validate_visdrone_yolo_structure
from src.data.yolo_label_reader import load_yolo_labels, yolo_bbox_to_xyxy_px
from src.evaluation.coco_converter import convert_yolo_gt_to_coco, convert_yolo_pred_txt_to_coco
from src.evaluation.coco_eval_runner import run_coco_eval
from src.evaluation.metrics_report import build_markdown_report
from src.policy.rule_engine import RuleEngineConfig, generate_policy_from_stats, save_policy_artifacts
from src.utils.io import dump_json, load_yaml

cfg = load_yaml(PROJECT_ROOT / 'configs' / 'project_config.yaml')
dataset_root = Path(cfg['dataset']['root'])
if not dataset_root.is_absolute():
    dataset_root = PROJECT_ROOT / dataset_root

TRAIN_IMAGES = 24
VAL_IMAGES = 8
ROOT = PROJECT_ROOT / 'artifacts' / 'visdrone_real_subset_smoke'
SUBSET_ROOT = ROOT / 'subset'
STATS_DIR = ROOT / 'stats'
POLICY_DIR = ROOT / 'policy'
EVAL_DIR = ROOT / 'eval'
REPORT_DIR = PROJECT_ROOT / 'artifacts' / 'reports'
for path in [STATS_DIR, POLICY_DIR, EVAL_DIR, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

assert dataset_root.exists(), f'VisDrone dataset not found: {dataset_root}. Set dataset.root in configs/project_config.yaml.'
assert (dataset_root / 'images' / 'train').exists(), 'Missing images/train'
assert (dataset_root / 'labels' / 'train').exists(), 'Missing labels/train'
assert (dataset_root / 'images' / 'val').exists(), 'Missing images/val'
assert (dataset_root / 'labels' / 'val').exists(), 'Missing labels/val'
print('VisDrone root =', dataset_root)

In [ ]:
timings = {}
t0 = time.perf_counter()
exts = tuple(cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp']))

start = time.perf_counter()
subset_report = build_yolo_subset(dataset_root, SUBSET_ROOT, train_images=TRAIN_IMAGES, val_images=VAL_IMAGES, seed=42, image_extensions=exts)
timings['subset_build'] = time.perf_counter() - start
assert subset_report.train_images > 0 and subset_report.val_images > 0, subset_report
print(subset_report)

In [ ]:
start = time.perf_counter()
validation = validate_visdrone_yolo_structure(SUBSET_ROOT, splits=('train', 'val'), image_extensions=exts)
timings['validation'] = time.perf_counter() - start
assert validation['is_valid'], validation
print(json.dumps(validation['splits'], indent=2, ensure_ascii=False))

In [ ]:
start = time.perf_counter()
stats = analyze_dataset(
    SUBSET_ROOT,
    output_dir=STATS_DIR,
    splits=('train', 'val'),
    config=DatasetAnalyzerConfig(
        small_max_area=float(cfg['analysis']['small_max_area']),
        medium_max_area=float(cfg['analysis']['medium_max_area']),
        tiny_max_area=float(cfg['analysis']['tiny_max_area']),
        image_extensions=exts,
        generate_plots=False,
        show_progress=False,
    ),
)
timings['analysis'] = time.perf_counter() - start
assert stats['splits']['train']['num_objects'] > 0
print('objects:', stats['splits']['train']['num_objects'])
print('small_ratio:', stats['splits']['train']['ratios']['small_ratio'])

In [ ]:
start = time.perf_counter()
policy, decision_report = generate_policy_from_stats(stats, RuleEngineConfig.from_project_config(cfg))
paths = save_policy_artifacts(policy, decision_report, POLICY_DIR)
timings['policy_generation'] = time.perf_counter() - start
assert policy['ultralytics_args']
assert policy['albumentations_spec']
print('fired rules:', [x['rule_name'] for x in decision_report['fired_rules']])

In [ ]:
import cv2

start = time.perf_counter()
bank = ObjectBank(small_max_area=float(cfg['analysis']['small_max_area']), tiny_max_area=float(cfg['analysis']['tiny_max_area']))
bank.build_from_dataset(SUBSET_ROOT / 'images' / 'train', SUBSET_ROOT / 'labels' / 'train', image_extensions=exts)
bank_path = POLICY_DIR / 'object_bank.json'
bank.save(bank_path)
timings['object_bank'] = time.perf_counter() - start
assert bank.size > 0

image_path = sorted((SUBSET_ROOT / 'images' / 'train').glob('*'))[0]
image = cv2.imread(str(image_path))
h, w = image.shape[:2]
labels = load_yolo_labels(SUBSET_ROOT / 'labels' / 'train' / f'{image_path.stem}.txt')
sample = {'image': image, 'bboxes': [list(yolo_bbox_to_xyxy_px(b, w, h)) for b in labels], 'class_labels': [b.class_id for b in labels]}
transforms = AugmentationPolicy(policy).get_albumentations_transforms(object_bank=bank, seed=42)
out = apply_custom_transforms(sample, transforms)
assert len(out['bboxes']) == len(out['class_labels'])
print('object_bank size:', bank.size, 'boxes after transforms:', len(out['bboxes']))

In [ ]:
start = time.perf_counter()
pred_dir = create_predictions_from_gt(SUBSET_ROOT / 'labels' / 'val', EVAL_DIR / 'pred_labels')
coco_gt = convert_yolo_gt_to_coco(SUBSET_ROOT / 'images' / 'val', SUBSET_ROOT / 'labels' / 'val', cfg['dataset']['visdrone_classes'], EVAL_DIR / 'coco_gt.json', image_extensions=exts)
coco_dt = convert_yolo_pred_txt_to_coco(pred_dir, SUBSET_ROOT / 'images' / 'val', EVAL_DIR / 'coco_dt.json', image_extensions=exts)
metrics = run_coco_eval(EVAL_DIR / 'coco_gt.json', EVAL_DIR / 'coco_dt.json', EVAL_DIR / 'coco_eval.json', use_tiny_eval=True, tiny_max_area=float(cfg['analysis']['tiny_max_area']))
timings['coco_eval'] = time.perf_counter() - start
assert len(coco_gt['images']) == subset_report.val_images
assert len(coco_dt) > 0
assert 'AP_small' in metrics
print(json.dumps(metrics, indent=2, ensure_ascii=False))

In [ ]:
timings['total'] = time.perf_counter() - t0
report = build_markdown_report(
    {'visdrone_real_subset_gt_predictions': metrics},
    output_path=REPORT_DIR / 'visdrone_real_subset_smoke_report.md',
    timings=timings,
    artifact_paths={
        'dataset_stats': str(STATS_DIR / 'dataset_stats.json'),
        'policy': str(POLICY_DIR / 'policy_adaptive.json'),
        'decision_report': str(POLICY_DIR / 'decision_report.json'),
        'object_bank': str(bank_path),
        'coco_eval': str(EVAL_DIR / 'coco_eval.json'),
    },
)
dump_json({'timings': timings, 'metrics': metrics}, REPORT_DIR / 'visdrone_real_subset_smoke_summary.json')
print('REAL VISDRONE SUBSET SMOKE TEST OK')
print('report =', report)
print(json.dumps(timings, indent=2))